# Detect Minino
The first step in the configuration process is to select the correct serial port to which the Minino device is connected.

<b>Identify the Port:</b> To check the available ports and determine the correct one, run the following code block:

In [1]:
import serial.tools.list_ports
import ipywidgets as widgets
from IPython.display import display, clear_output

available_ports = []
scan_ports = widgets.Button(description="Scan ports")

ports_dropdown = widgets.Dropdown(
    options=available_ports,
    description='Ports:',
    layout=widgets.Layout(width='20%')
)

def detect_serial_ports(btn):
    # limpiamos la lista
    available_ports.clear()

    ports = serial.tools.list_ports.comports()
    for port in ports:
        if any(x in port.device for x in ["ttyUSB", "ttyACM", "cu.usbmodem", "usbserial", "COM"]):
            available_ports.append(port.device)

    # actualizar el dropdown
    ports_dropdown.options = available_ports

detect_serial_ports(None)

scan_ports.on_click(detect_serial_ports)

display(scan_ports, ports_dropdown)

Button(description='Scan ports', style=ButtonStyle())

Dropdown(description='Ports:', layout=Layout(width='20%'), options=('COM1',), value=None)

# Hands on Lab #4  Minino - Wifi Captive Portal
The Wifi Captive Portal enables the Minino device to function as a Wi-Fi Access Point (AP), allowing it to intercept client connections and present a fully customizable web interface. This brings enhanced flexibility, user data handling, and configuration options, making it easier to create engaging and effective captive experiences.

## Objetives

- <b>Explain the concept</b> of a rogue access point and its use in credential harvesting attacks.
- <b>Configure Minino</b> to host a captive portal that mimics a legitimate Wi‑Fi network.
- <b>Capture HTTP credentials</b> and probe requests from automatically connecting devices.
- <b>Analyze</b> the behavior of client devices during rogue AP interactions.
- <b>Log</b> and interpret the captured credential data for reporting or forensic purposes.
- <b>Evaluate</b> the security risks posed by untrusted Wi‑Fi networks and rogue APs.

#### What Is an Access Point?
A Wi-Fi access point (AP) is a networking device that allows wireless devices to connect to a wired network using Wi-Fi.

#### A Rogue Access Point
A rogue access point is an unauthorized wireless access point connected to a private network, creating a significant security risk. These can be set up by malicious actors or even by employees without permission, providing a backdoor for attackers to steal data, spread malware, or launch further attacks

### Check if the connected device at the port is a Minino
Before proceeding, it is necessary to verify that the device connected to the selected serial port is indeed the Minino device.
Minino is configured to transmit a specific message (or "signature") upon connection. This message serves as a positive indicator to confirm the device identity.

In [3]:
# Detect the Minino Board
import serial, time

output_serial = widgets.Output()
label_port = widgets.Label(value = f"PORT SELECTED {ports_dropdown.value}")
detect_minino_button = widgets.Button(description="Detect Minino")

serial_port = ports_dropdown.value
serial_conn = None
is_a_minino_board = False

# Function to connect the Serial
def connect_serial():
    global serial_conn
    # Connect with the port
    try:
        serial_conn = serial.Serial(serial_port, 115200, timeout=1)
        time.sleep(2)
        return True
    except Exception as e:
        return False

# Function to send a command via Serial with a response 
def send_command_string_with_response(command:str):
    global serial_conn

    # Send any word to get the return
    if serial_conn and serial_conn.is_open:
        cmd = command+'\r\n'
        serial_conn.write(cmd.encode())
        time.sleep(0.3)
        try: 
            resp = serial_conn.read_all().decode(errors='ignore')
            return resp
        except:
            return 'Nan'
            
    else:
        return 'Nan'

# Function to disconnect Serial
def disconnect_serial():
    global serial_conn
    
    # Disconnect with the port
    if serial_conn and serial_conn.is_open:
        serial_conn.close()
        serial_conn = None
        return True
    else: 
        return False

# Function to detect the minino
def detect_minino(btn):
    connect_serial()
    data = send_command_string_with_response('h')
    if "minino" in data:
        is_a_minino_board = True

        with output_serial:
            clear_output() 
            print(f"Minino Detected at {serial_port}")
    else:
        with output_serial:
            clear_output()
            print("Minino no detected")
    disconnect_serial()

# Test for minino
detect_minino_button.on_click(detect_minino)
display(label_port,detect_minino_button, output_serial)

Label(value='PORT SELECTED COM34')

Button(description='Detect Minino', style=ButtonStyle())

Output()

### Naming the Captive Portal
The <b>SSID</b> (Service Set Identifier) is the human-readable name you assign to the Wi-Fi network (Access Point) hosted by Minino via the USB interface.
You could provide an SSID for the Access Point (AP). When the Captive Portal is executed, Minino will create a Wi-Fi AP using this assigned name, allowing client devices to connect to it.
Execute the following block of code, it will display a button and text box, write the name for the SSID and configure the name clicking on the button 

In [4]:
button_send_name = widgets.Button(description="send name")
text_input = widgets.Text()
respond_serial = widgets.Output()

def send_captive_name(b=None):

    with respond_serial:
        clear_output()  # Clear display

        name = text_input.value
        if not name:
            print("Please write a name")
            return

        # Connect Serial 
        global serial_conn
        if not serial_conn:
            if not connect_serial():
                print("ERROR while connecting serial")
                return
            else:
                print(f"Device at {serial_port} connected")

        # First send a help
        send_command_string_with_response("help")

        command_complete = f"captive_name {name}"
        print("Sending:", command_complete)

        response = send_command_string_with_response(command_complete)

        print("Response:", response)

        # Disconnect Serial
        if disconnect_serial():
            print(f"Device at {serial_port} disconnected")
        else:
            print("ERROR AT DISCONNECT")

button_send_name.on_click(send_captive_name)

display(widgets.HBox([text_input, button_send_name]), respond_serial)

Output()

### Running the Captive Portal
<b>Launch the Captive Portal Menu:</b> Minino does not feature a direct command to immediately generate the Wi-Fi AP. Instead, you must first execute the command that launches the configuration menu for the Captive Portal feature. On the next block of code, after the execution, it will send you to the Captive Portal configuration menu there navigate to the Run option using the buttons (up/down) and execute the option by pressing the right button (or the designated execution button).

In [5]:
connect_button = widgets.Button(description="Connect", button_style='info')
disconnect_button = widgets.Button(description="Disconnect", button_style='danger')
go_to_captive_portal_button = widgets.Button(description="Captive Portal", button_style='')

captive_output = widgets.Output()

# Connect with the button
def connect_button_fn(b=None):

    global serial_conn
    
    if not serial_conn:
        if not connect_serial():
            with captive_output:
                print("Erro Conecting")
            return
            
    with captive_output:
        print(f"Device connected at {serial_port}")

# Send the command
def captive_portal_fn(b = None):

    global serial_conn

    response = ""
    
    if serial_conn and serial_conn.is_open:
        response = send_command_string_with_response("launch captive_portal")
    else:
        with captive_output:
            print("Error")
        return

    with captive_output:
        print(f"{response}")
            
    

#Disconnect Minino
def disconnect_button_fn(b = None):
    if disconnect_serial():
        with captive_output:
            print(f"Device at {serial_port} disconnected")
    else:
        with captive_output:
            print("ERROR AT DISCONNECT")

# Set functions on the buttons
connect_button.on_click(connect_button_fn)
disconnect_button.on_click(disconnect_button_fn)
go_to_captive_portal_button.on_click(captive_portal_fn)

# Display Buttons
display(widgets.HBox([connect_button,go_to_captive_portal_button,disconnect_button]), captive_output)

Output()

### Verification: Checking the Active AP
Once the Captive Portal has been executed, you must verify that the Access Point (AP) is successfully broadcasting the signal.

You can check this using any Wi-Fi enabled device:
- Scan for Networks: Use your mobile phone or computer to scan the surrounding Wi-Fi networks.
- Locate the SSID: You should see a new Access Point listed with the SSID name you previously configured for the Minino device.

If the network appears, the Captive Portal is active and ready to intercept client connections.

## Quick references
- **Captive Portal:** Unauthorized Wi‑Fi access point set up to trick users into connecting, often for credential capture or traffic interception.
- **Rogue Access Point:** Web page presented to users before granting internet access; in attacks, it can be used to harvest credentials.
- **Probe Request:** Wi‑Fi frame sent by a client to discover known networks; can reveal preferred SSIDs and past connections.
- **HTTP Credentials:** Usernames, passwords, or other authentication data sent in plaintext via HTTP; easily intercepted if unencrypted.
- **SSID:** Network name broadcast by a Wi‑Fi access point.